# Review Agent Test Notebook

## Setup

In [ ]:
# Install required packages
# %pip install langchain langchain-openai openai python-dotenv

In [ ]:
import sys
import os

module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

In [ ]:
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

client = OpenAI()

model = os.getenv("LLM_MODEL_NAME", "gpt-5o-mini")

print(f"✅ OpenAI client initialized with model '{model}'")

In [ ]:
from utils.logging_utils import init_logs, log_openai_response, get_logs_dataframe, get_summary, list_log_files, log_langchain_response

RUN_ID = os.getenv("RUN_ID", "default_run")

init_logs(RUN_ID)

print("✅ Logging initialized")
print(f"RUN_ID: {RUN_ID}")

In [ ]:
# Load generated content and style guides
import json

with open("../artifact/swift_announcement_corpus.json", "r") as f:
    corpus_data = json.load(f)

with open("../artifact/style_guides.json", "r") as f:
    style_guides = json.load(f)

print("✅ Loaded corpus and style guides")
print(f"BBC draft length: {len(corpus_data['bbc']['content'])} chars")
print(f"Taylor Swift draft length: {len(corpus_data['taylor_swift']['content'])} chars")

In [ ]:
# Load review models
from utils.review_models import ReviewOutput, StyleViolation

print("✅ Review models loaded")

## Review Prompts

In [ ]:
BBC_REVIEW_PROMPT = """You are a Professional BBC Copy Editor.

OBJECTIVE:
Cross-reference the [DRAFT] against the provided [BBC STYLE DNA].

INSTRUCTIONS:
1. Check 'Tone' - Should be neutral, factual, impartial
2. Check 'Structure' - Should use Inverted Pyramid (most important first), headline + lead paragraph
3. Check 'Punctuation' - Headlines use "Verb + Object + Colon" pattern, proper quotation marks
4. Check 'Vocabulary' - Plain-language journalism, accessible register
5. Check 'Key Phrases' - Look for: "has hit", "Quarterly profits", "said", "on Friday"

OUTPUT:
Provide your review in the structured JSON format with:
- overall_fidelity_score (1-10)
- violations (list of aspect, issue, suggestion)
- critique_summary

Be specific so the human editor knows exactly what to change."""

In [ ]:
TS_REVIEW_PROMPT = """You are a Professional Copy Editor for Taylor Swift's PR team.

OBJECTIVE:
Cross-reference the [DRAFT] against the provided [TAYLOR SWIFT STYLE DNA].

INSTRUCTIONS:
1. Check 'Tone' - Should be conversational, intimate, first-person, emotional
2. Check 'Structure' - Short fragments, rhetorical questions, line breaks for pacing
3. Check 'Punctuation' - Lowercase for aesthetic, emojis, repeated punctuation (!!!)
4. Check 'Formatting' - Multi-line tweets, parenthesized dates, @mentions
5. Check 'Key Phrases' - Look for: "Pre-order now", "Meet me at midnight", "I AM IN SHAMBLES"

OUTPUT:
Provide your review in the structured JSON format with:
- overall_fidelity_score (1-10)
- violations (list of aspect, issue, suggestion)
- critique_summary

Be specific so the human editor knows exactly what to change."""

## Review Function

In [ ]:
from langchain_openai import ChatOpenAI

# Create LangChain LLM with structured output
llm = ChatOpenAI(model=model, temperature=0.3)
structured_llm = llm.with_structured_output(ReviewOutput)

def review_content(draft: str, style_dna: str, source_type: str) -> dict:
    """
    Review content against style DNA.
    
    Args:
        draft: The AI-generated content to review
        style_dna: The style guide to compare against
        source_type: "bbc" or "taylor_swift"
    
    Returns:
        Dictionary with review output
    """
    # Select appropriate prompt
    if source_type == "bbc":
        system_prompt = BBC_REVIEW_PROMPT
    else:
        system_prompt = TS_REVIEW_PROMPT

    # Build the full prompt
    full_prompt = f"""{system_prompt}

=== DRAFT ===
{draft}

=== BBC STYLE DNA ===
{style_dna}"""

    task_name = f"{source_type}_Reviewer"

    # Call LLM with structured output
    response = structured_llm.invoke(full_prompt)
    result = log_langchain_response(response, full_prompt, task_name=task_name, run_id=RUN_ID)

    return result

print("✅ Review function defined")

## Test: Review BBC Article

In [ ]:
# Get BBC data
bbc_draft = corpus_data["bbc"]["content"]
bbc_topic = corpus_data["bbc"]["topic"]
bbc_style_dna = style_guides["bbc"]

print(f"Topic: {bbc_topic}")
print("-" * 50)

# Review BBC
bbc_review_result = review_content(bbc_draft, bbc_style_dna, "bbc")

print(f"Overall Fidelity Score: {bbc_review_result['overall_fidelity_score']}/10")
print(f"\nCritique Summary: {bbc_review_result['critique_summary']}")
print(f"\nViolations ({len(bbc_review_result['violations'])}):")
for v in bbc_review_result['violations']:
    print(f"  - {v['aspect']}: {v['issue']}")
    print(f"    Suggestion: {v['suggestion']}")

## Test: Review Taylor Swift Tweet

In [ ]:
# Get Taylor Swift data
ts_draft = corpus_data["taylor_swift"]["content"]
ts_topic = corpus_data["taylor_swift"]["topic"]
ts_style_dna = style_guides["taylor_swift"]

print(f"Topic: {ts_topic}")
print("-" * 50)

# Review Taylor Swift
ts_review_result = review_content(ts_draft, ts_style_dna, "taylor_swift")

print(f"Overall Fidelity Score: {ts_review_result['overall_fidelity_score']}/10")
print(f"\nCritique Summary: {ts_review_result['critique_summary']}")
print(f"\nViolations ({len(ts_review_result['violations'])}):")
for v in ts_review_result['violations']:
    print(f"  - {v['aspect']}: {v['issue']}")
    print(f"    Suggestion: {v['suggestion']}")

## Save to suggested_changes.json

In [ ]:
# Save review results to JSON

# BBC review
bbc_suggested_changes = {
    "run_id": RUN_ID,
    "ai_draft": bbc_draft,
    "topic": bbc_topic,
    "style_dna": bbc_style_dna,
    "reviewer_notes": {
        "overall_fidelity_score": bbc_review_result["overall_fidelity_score"],
        "violations": bbc_review_result["violations"],
        "critique_summary": bbc_review_result["critique_summary"]
    },
    "user_modified_content": "",
    "status": "AWAITING_HUMAN_EDIT"
}

# Taylor Swift review
ts_suggested_changes = {
    "run_id": RUN_ID,
    "ai_draft": ts_draft,
    "topic": ts_topic,
    "style_dna": ts_style_dna,
    "reviewer_notes": {
        "overall_fidelity_score": ts_review_result["overall_fidelity_score"],
        "violations": ts_review_result["violations"],
        "critique_summary": ts_review_result["critique_summary"]
    },
    "user_modified_content": "",
    "status": "AWAITING_HUMAN_EDIT"
}

# Save to artifact folder
os.makedirs("../artifact", exist_ok=True)

# Save BBC
with open("../artifact/suggested_changes_bbc.json", "w") as f:
    json.dump(bbc_suggested_changes, f, indent=2)

# Save Taylor Swift
with open("../artifact/suggested_changes_taylor_swift.json", "w") as f:
    json.dump(ts_suggested_changes, f, indent=2)

print("✅ Saved review results:")
print("  - artifact/suggested_changes_bbc.json")
print("  - artifact/suggested_changes_taylor_swift.json")